In [1]:
!pip install boto3

In [2]:
!pip install netCDF4
!pip install h5netcdf

In [3]:
# import dependencies
import pandas as pd
import os
import requests
import json
import xarray as xr
import boto3

In [4]:
# Load Our World in Data CO2 data ('owid-co2-data.csv')
owid_co2_df = pd.read_csv('Resources/owid-co2-data.csv')
owid_co2_df.head(20)

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Afghanistan,1755,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Afghanistan,1756,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Afghanistan,1757,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Afghanistan,1758,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Afghanistan,1759,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


source: https://github.com/owid/co2-data?tab=readme-ov-file

In [5]:
# Look at columns of owid_co2_df
owid_co2_df.columns

Index(['country', 'year', 'iso_code', 'population', 'gdp', 'cement_co2',
       'cement_co2_per_capita', 'co2', 'co2_growth_abs', 'co2_growth_prct',
       'co2_including_luc', 'co2_including_luc_growth_abs',
       'co2_including_luc_growth_prct', 'co2_including_luc_per_capita',
       'co2_including_luc_per_gdp', 'co2_including_luc_per_unit_energy',
       'co2_per_capita', 'co2_per_gdp', 'co2_per_unit_energy', 'coal_co2',
       'coal_co2_per_capita', 'consumption_co2', 'consumption_co2_per_capita',
       'consumption_co2_per_gdp', 'cumulative_cement_co2', 'cumulative_co2',
       'cumulative_co2_including_luc', 'cumulative_coal_co2',
       'cumulative_flaring_co2', 'cumulative_gas_co2', 'cumulative_luc_co2',
       'cumulative_oil_co2', 'cumulative_other_co2', 'energy_per_capita',
       'energy_per_gdp', 'flaring_co2', 'flaring_co2_per_capita', 'gas_co2',
       'gas_co2_per_capita', 'ghg_excluding_lucf_per_capita', 'ghg_per_capita',
       'land_use_change_co2', 'land_use_chang

In [6]:
# Load Cost and Affordability of a Healthy Diet (CoAHD) dataset ('FAO_CAHD.csv')
healthy_diet_cost_df = pd.read_csv('Resources/FAO_CAHD.csv')

# See all the columns
pd.set_option("display.max_columns", None)

# See truncated entries
pd.set_option("display.max_colwidth", None)

healthy_diet_cost_df.head()

,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,SEX,SEX_LABEL,AGE,AGE_LABEL,URBANISATION,URBANISATION_LABEL,UNIT_MEASURE,UNIT_MEASURE_LABEL,COMP_BREAKDOWN_1,COMP_BREAKDOWN_1_LABEL,COMP_BREAKDOWN_2,COMP_BREAKDOWN_2_LABEL,COMP_BREAKDOWN_3,COMP_BREAKDOWN_3_LABEL,TIME_PERIOD,OBS_VALUE,DATABASE_ID,DATABASE_ID_LABEL,UNIT_MULT,UNIT_MULT_LABEL,UNIT_TYPE,UNIT_TYPE_LABEL,TIME_FORMAT,TIME_FORMAT_LABEL,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL
0,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,ALB,Albania,FAO_CAHD_7005,Percentage of the population unable to afford a healthy diet (percent),_T,Total,_T,All age ranges or no breakdown by age,_T,Total,PT,Percentage,_Z,Not Applicable,_Z,Not Applicable,_Z,Not Applicable,2017,24.3,FAO_CAHD,Cost and Affordability of a Healthy Diet (CoAHD),0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
1,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,ALB,Albania,FAO_CAHD_7005,Percentage of the population unable to afford a healthy diet (percent),_T,Total,_T,All age ranges or no breakdown by age,_T,Total,PT,Percentage,_Z,Not Applicable,_Z,Not Applicable,_Z,Not Applicable,2018,17.6,FAO_CAHD,Cost and Affordability of a Healthy Diet (CoAHD),0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
2,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,ALB,Albania,FAO_CAHD_7005,Percentage of the population unable to afford a healthy diet (percent),_T,Total,_T,All age ranges or no breakdown by age,_T,Total,PT,Percentage,_Z,Not Applicable,_Z,Not Applicable,_Z,Not Applicable,2019,14.6,FAO_CAHD,Cost and Affordability of a Healthy Diet (CoAHD),0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
3,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,ALB,Albania,FAO_CAHD_7005,Percentage of the population unable to afford a healthy diet (percent),_T,Total,_T,All age ranges or no breakdown by age,_T,Total,PT,Percentage,_Z,Not Applicable,_Z,Not Applicable,_Z,Not Applicable,2020,13.9,FAO_CAHD,Cost and Affordability of a Healthy Diet (CoAHD),0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
4,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,ALB,Albania,FAO_CAHD_7005,Percentage of the population unable to afford a healthy diet (percent),_T,Total,_T,All age ranges or no breakdown by age,_T,Total,PT,Percentage,_Z,Not Applicable,_Z,Not Applicable,_Z,Not Applicable,2021,12.6,FAO_CAHD,Cost and Affordability of a Healthy Diet (CoAHD),0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public


source: https://data360.worldbank.org/en/dataset/FAO_CAHD

In [7]:
# Let's get a list of all these column headers
healthy_diet_cost_df.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'FREQ', 'FREQ_LABEL', 'REF_AREA',
       'REF_AREA_LABEL', 'INDICATOR', 'INDICATOR_LABEL', 'SEX', 'SEX_LABEL',
       'AGE', 'AGE_LABEL', 'URBANISATION', 'URBANISATION_LABEL',
       'UNIT_MEASURE', 'UNIT_MEASURE_LABEL', 'COMP_BREAKDOWN_1',
       'COMP_BREAKDOWN_1_LABEL', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_2_LABEL',
       'COMP_BREAKDOWN_3', 'COMP_BREAKDOWN_3_LABEL', 'TIME_PERIOD',
       'OBS_VALUE', 'DATABASE_ID', 'DATABASE_ID_LABEL', 'UNIT_MULT',
       'UNIT_MULT_LABEL', 'UNIT_TYPE', 'UNIT_TYPE_LABEL', 'TIME_FORMAT',
       'TIME_FORMAT_LABEL', 'OBS_STATUS', 'OBS_STATUS_LABEL', 'OBS_CONF',
       'OBS_CONF_LABEL'],
      dtype='object')

In [8]:
# List of preferred columns
columns_list = [
    'REF_AREA_LABEL', 'INDICATOR_LABEL', 'AGE_LABEL', 'URBANISATION_LABEL', 'UNIT_MEASURE_LABEL', 'UNIT_MULT', 'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS_LABEL'
]

In [9]:
# New condensed df
edited_diet_df = healthy_diet_cost_df[columns_list]
edited_diet_df.head()

,REF_AREA_LABEL,INDICATOR_LABEL,AGE_LABEL,URBANISATION_LABEL,UNIT_MEASURE_LABEL,UNIT_MULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS_LABEL
0,Albania,Percentage of the population unable to afford a healthy diet (percent),All age ranges or no breakdown by age,Total,Percentage,0,2017,24.3,Normal value
1,Albania,Percentage of the population unable to afford a healthy diet (percent),All age ranges or no breakdown by age,Total,Percentage,0,2018,17.6,Normal value
2,Albania,Percentage of the population unable to afford a healthy diet (percent),All age ranges or no breakdown by age,Total,Percentage,0,2019,14.6,Normal value
3,Albania,Percentage of the population unable to afford a healthy diet (percent),All age ranges or no breakdown by age,Total,Percentage,0,2020,13.9,Normal value
4,Albania,Percentage of the population unable to afford a healthy diet (percent),All age ranges or no breakdown by age,Total,Percentage,0,2021,12.6,Normal value


In [10]:
# Look at unique values in INDICATOR_LABEL column
edited_diet_df['INDICATOR_LABEL'].unique()

array(['Percentage of the population unable to afford a healthy diet (percent)',
       'Number of people unable to afford a healthy diet (million)'],
      dtype=object)

In [11]:
# Look at unique values in AGE_LABEL column
edited_diet_df['AGE_LABEL'].unique()

array(['All age ranges or no breakdown by age'], dtype=object)

In [12]:
# Look at unique values in URBANISATION_LABEL column
edited_diet_df['URBANISATION_LABEL'].unique()

array(['Total'], dtype=object)

In [13]:
# Look at unique values in UNIT_MEASURE_LABEL column
edited_diet_df['UNIT_MEASURE_LABEL'].unique()

array(['Percentage', 'Persons'], dtype=object)

In [14]:
# Look at unique values in OBS_STATUS_LABEL column
edited_diet_df['OBS_STATUS_LABEL'].unique()

array(['Normal value'], dtype=object)

'AGE_LABEL', 'URBANISATION LABEL', 'OBS_STATUS_LABEL' columns only have one unique entry. 'AGE_LABEL' means all ages are included, this column can be dropped. 'URBANISATION_LABEL' is always "Total", this column can be dropped. 'OBS_STATUS_LABEL' is always "Normal", this column can be dropped.

In [15]:
# Drop unnecessary columns
edited_diet_df = edited_diet_df.drop(['AGE_LABEL', 'URBANISATION_LABEL', 'OBS_STATUS_LABEL'],
                                     axis=1
                                     )

In [16]:
edited_diet_df.head()

,REF_AREA_LABEL,INDICATOR_LABEL,UNIT_MEASURE_LABEL,UNIT_MULT,TIME_PERIOD,OBS_VALUE
0,Albania,Percentage of the population unable to afford a healthy diet (percent),Percentage,0,2017,24.3
1,Albania,Percentage of the population unable to afford a healthy diet (percent),Percentage,0,2018,17.6
2,Albania,Percentage of the population unable to afford a healthy diet (percent),Percentage,0,2019,14.6
3,Albania,Percentage of the population unable to afford a healthy diet (percent),Percentage,0,2020,13.9
4,Albania,Percentage of the population unable to afford a healthy diet (percent),Percentage,0,2021,12.6


In [17]:
# Look at value counts for INDICATOR_LABEL column
edited_diet_df['INDICATOR_LABEL'].value_counts()

INDICATOR_LABEL
Percentage of the population unable to afford a healthy diet (percent)    1202
Number of people unable to afford a healthy diet (million)                1122
Name: count, dtype: int64

In [18]:
# Look at value counts for UNIT_MEASURES_LABEL
edited_diet_df['UNIT_MEASURE_LABEL'].value_counts()

UNIT_MEASURE_LABEL
Percentage    1202
Persons       1122
Name: count, dtype: int64

In [19]:
# Drop INDICATOR_LABEL column, as it coincides directly with UNIT_MEASURE_LABEL and is more cumbersome
edited_diet_df = edited_diet_df.drop(['INDICATOR_LABEL'], axis=1)

In [20]:
edited_diet_df.head()

,REF_AREA_LABEL,UNIT_MEASURE_LABEL,UNIT_MULT,TIME_PERIOD,OBS_VALUE
0,Albania,Percentage,0,2017,24.3
1,Albania,Percentage,0,2018,17.6
2,Albania,Percentage,0,2019,14.6
3,Albania,Percentage,0,2020,13.9
4,Albania,Percentage,0,2021,12.6


Because they use different units of measurement, it's hard to know what to do. Roughly half the dataset uses different units. It would be possible to manually calculate the percentage or number of persons (convert the data to one or the other units). Or maybe just drop the entries counted in 'Persons'. Let's look at an example of a row counted in 'Persons'.

In [21]:
# Look at value counts for UNIT_MULT column
edited_diet_df['UNIT_MULT'].value_counts()

UNIT_MULT
0    1202
6    1122
Name: count, dtype: int64

In [22]:
# Find a persons row
row = edited_diet_df[edited_diet_df['UNIT_MEASURE_LABEL'] == "Persons"]
row

,REF_AREA_LABEL,UNIT_MEASURE_LABEL,UNIT_MULT,TIME_PERIOD,OBS_VALUE
1202,Albania,Persons,6,2017,0.7
1203,Albania,Persons,6,2018,0.5
1204,Albania,Persons,6,2019,0.4
1205,Albania,Persons,6,2020,0.4
1206,Albania,Persons,6,2021,0.4
...,...,...,...,...,...
2319,Ghana,Persons,6,2023,22.4
2320,Greece,Persons,6,2024,1.7
2321,Armenia,Persons,6,2024,1.9
2322,Armenia,Persons,6,2023,1.5


In [23]:
# Look at REF_AREA_LABEL value counts
edited_diet_df['REF_AREA_LABEL'].value_counts()

REF_AREA_LABEL
Albania      16
Jordan       16
Myanmar      16
Namibia      16
Nepal        16
             ..
Malta         8
St. Lucia     8
Argentina     2
Zimbabwe      2
Qatar         1
Name: count, Length: 153, dtype: int64

In [24]:
# Look at TIME_PERIOD value counts
edited_diet_df['TIME_PERIOD'].value_counts()

TIME_PERIOD
2017    297
2018    292
2023    291
2019    290
2020    290
2021    289
2022    289
2024    286
Name: count, dtype: int64

Open Aquastat CSV file (FAO-AS.csv) from: https://data360.worldbank.org/en/dataset/FAO_AS

Was hoping for rainfall data, doesn't seem to be in here.

In [26]:
# Load ClimateWatch dataset from
climate_df = pd.read_csv('Resources/WRI_CLIMATEWATCH.csv')
climate_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Resources/WRI_CLIMATEWATCH.csv'

In [ ]:
# Look at climate_head columns
climate_df.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'FREQ', 'FREQ_LABEL', 'REF_AREA',
       'REF_AREA_LABEL', 'INDICATOR', 'INDICATOR_LABEL', 'SEX', 'SEX_LABEL',
       'AGE', 'AGE_LABEL', 'URBANISATION', 'URBANISATION_LABEL',
       'UNIT_MEASURE', 'UNIT_MEASURE_LABEL', 'COMP_BREAKDOWN_1',
       'COMP_BREAKDOWN_1_LABEL', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_2_LABEL',
       'COMP_BREAKDOWN_3', 'COMP_BREAKDOWN_3_LABEL', 'TIME_PERIOD',
       'OBS_VALUE', 'DATABASE_ID', 'DATABASE_ID_LABEL', 'UNIT_MULT',
       'UNIT_MULT_LABEL', 'UNIT_TYPE', 'UNIT_TYPE_LABEL', 'TIME_FORMAT',
       'TIME_FORMAT_LABEL', 'OBS_STATUS', 'OBS_STATUS_LABEL', 'OBS_CONF',
       'OBS_CONF_LABEL'],
      dtype='object')

In [ ]:
# Look at unique values of "INDICATOR_LABEL" column
climate_df['INDICATOR_LABEL'].value_counts()

INDICATOR_LABEL
Greenhouse gas (GHG) emissions caused by the transportation sector    12202
Name: count, dtype: int64

In [ ]:
# Load data dictionary for climate_df
climate_dict_df = pd.read_csv('Resources/WRI_CLIMATEWATCH_DATADICT.csv')
climate_dict_df

,VARIABLE_ID,VARIABLE_LABEL,VARIABLE_DESCRIPTION,VARIABLE_DATA_TYPE,VARIABLE_REQUIRED
0,FREQ,Frequency of observation,Time interval at which observations occur over a given time period.,String,True
1,FREQ_LABEL,Frequency of observation,Time interval at which observations occur over a given time period.,String,True
2,REF_AREA,Reference area,Country or geographic area to which the measured statistical phenomenon relates.,String,True
3,REF_AREA_LABEL,Reference area,Country or geographic area to which the measured statistical phenomenon relates.,String,True
4,INDICATOR,Statistical indicator,"Data element that represents statistical data for a specified time, place, and other characteristics, and is corrected for at least one dimension (usually size) to allow for meaningful comparisons.",String,True
5,INDICATOR_LABEL,Statistical indicator,"Data element that represents statistical data for a specified time, place, and other characteristics, and is corrected for at least one dimension (usually size) to allow for meaningful comparisons.",String,True
6,SEX,Sex,State of being male or female.,String,True
7,SEX_LABEL,Sex,State of being male or female.,String,True
8,AGE,Age,Length of time that an entity has lived or existed.,String,True
9,AGE_LABEL,Age,Length of time that an entity has lived or existed.,String,True


Source: https://data360.worldbank.org/en/dataset/WRI_CLIMATEWATCH

Some files are too large (FAO_AS.csv) to upload to Github. Maybe use an s3 bucket eventually.

Source: World Bank, Climate Change Knowledge Portal (2025). URL: https://climateknowledgeportal.worldbank.org/. Date Accessed: 

In [27]:
# Get Climate Change Knowledge Portal API dataset about rainfall in North America

# Save url
url = 'https://cckpapi.worldbank.org/cckp/v1/cru-x0.5_timeseries_pr_timeseries_annual_1901-2024_mean_historical_cru_ts4.09_mean/region_nac?_format=json'

In [28]:
# Call the API
response = requests.get(url)
response.raise_for_status()

In [29]:
# Convert to Python dict
data = response.json()

In [30]:
# Extract the nested 'data' dict
nested_data = data['data']

In [31]:
# Build a list with data
records = []
for region_code, timeseries in nested_data.items():
    if timeseries is None:
        continue  # skip regions with no data
    for date, value in timeseries.items():
        records.append({
            "region_code": region_code,
            "date": date,
            "rainfall_mm": value
        })


In [32]:
# Now try to create a DataFrame
rainfall_df = pd.DataFrame(records)

In [33]:
# Check on the df
rainfall_df.head()

,region_code,date,rainfall_mm
0,ABW,1901-07,420.9
1,ABW,1902-07,420.9
2,ABW,1903-07,420.9
3,ABW,1904-07,420.9
4,ABW,1905-07,420.9


In [34]:
# Check rainfall_df shape
rainfall_df.shape

(622976, 3)

In [35]:
# Save rainfall_df to CSV
rainfall_df.to_csv('Resources/NA_rainfall_timeseries.csv', index=False)

In [36]:
# Let's look at unique region codes
regions = list(rainfall_df['region_code'].unique())
regions

['ABW',
 'ABW.14465',
 'AFG',
 'AFG.11',
 'AFG.110',
 'AFG.111',
 'AFG.112',
 'AFG.113',
 'AFG.114',
 'AFG.115',
 'AFG.116',
 'AFG.117',
 'AFG.118',
 'AFG.119',
 'AFG.12',
 'AFG.120',
 'AFG.121',
 'AFG.122',
 'AFG.123',
 'AFG.124',
 'AFG.125',
 'AFG.126',
 'AFG.127',
 'AFG.128',
 'AFG.129',
 'AFG.13',
 'AFG.130',
 'AFG.131',
 'AFG.132',
 'AFG.133',
 'AFG.134',
 'AFG.14',
 'AFG.15',
 'AFG.16',
 'AFG.17',
 'AFG.18',
 'AFG.19',
 'AGO',
 'AGO.8398',
 'AGO.8399',
 'AGO.8400',
 'AGO.8401',
 'AGO.8402',
 'AGO.8403',
 'AGO.8404',
 'AGO.8405',
 'AGO.8406',
 'AGO.8407',
 'AGO.8408',
 'AGO.8409',
 'AGO.8410',
 'AGO.8411',
 'AGO.8412',
 'AGO.8413',
 'AGO.8414',
 'AGO.8415',
 'AIA',
 'AIA.9416',
 'AIA.9417',
 'AIA.9418',
 'AIA.9419',
 'AIA.9420',
 'AIA.9421',
 'AIA.9422',
 'AIA.9425',
 'ALA',
 'ALA.841242',
 'ALB',
 'ALB.3305',
 'ALB.3306',
 'ALB.3307',
 'ALB.3308',
 'ALB.3309',
 'ALB.3310',
 'ALB.3311',
 'ALB.3312',
 'ALB.3313',
 'ALB.3314',
 'ALB.3315',
 'ALB.3316',
 'ALB.3317',
 'ALB.3318',
 'AL

In [37]:
# Try to get only North American data
north_america_codes = ['CAN', 'USA']  # add more if needed
rainfall_df_na = rainfall_df[rainfall_df['region_code'].str.startswith(tuple(north_america_codes))]


In [38]:
rainfall_df_na.head()

,region_code,date,rainfall_mm
54064,CAN,1901-07,538.38
54065,CAN,1902-07,549.26
54066,CAN,1903-07,540.08
54067,CAN,1904-07,531.71
54068,CAN,1905-07,531.56


In [39]:
# Look at rainfall_na_df
rainfall_df_na.shape

(8184, 3)

In [40]:
# Check for na
rainfall_df_na.isna().sum()

region_code    0
date           0
rainfall_mm    0
dtype: int64

In [41]:
# Check unique entries in 'region_code'
rainfall_df_na['region_code'].unique()

array(['CAN', 'CAN.461', 'CAN.4610', 'CAN.4611', 'CAN.4612', 'CAN.4613',
       'CAN.462', 'CAN.463', 'CAN.464', 'CAN.465', 'CAN.466', 'CAN.467',
       'CAN.468', 'CAN.469', 'USA', 'USA.2593214', 'USA.2593215',
       'USA.2593216', 'USA.2593217', 'USA.2593218', 'USA.2593219',
       'USA.2593220', 'USA.2593221', 'USA.2593222', 'USA.2593223',
       'USA.2593224', 'USA.2593225', 'USA.2593226', 'USA.2593227',
       'USA.2593228', 'USA.2593229', 'USA.2593230', 'USA.2593231',
       'USA.2593232', 'USA.2593233', 'USA.2593234', 'USA.2593235',
       'USA.2593236', 'USA.2593237', 'USA.2593238', 'USA.2593239',
       'USA.2593240', 'USA.2593241', 'USA.2593242', 'USA.2593243',
       'USA.2593244', 'USA.2593245', 'USA.2593246', 'USA.2593247',
       'USA.2593248', 'USA.2593249', 'USA.2593250', 'USA.2593251',
       'USA.2593252', 'USA.2593253', 'USA.2593254', 'USA.2593255',
       'USA.2593256', 'USA.2593257', 'USA.2593258', 'USA.2593259',
       'USA.2593260', 'USA.2593261', 'USA.2593262',

Let's use the provided raster file to understand what these regions are.

It was uploaded to an s3 bucket, first let's retrieve it from there.

In [42]:
# Get s3 url
url = "https://portfolio-5867-9443-9985.s3.us-east-2.amazonaws.com/timeseries-pr-annual-mean_cru-x0.5_cru-ts4.09-historical_timeseries_mean_1901-2024.nc"


In [43]:
# Save NetCDF file to disk using streaming
with requests.get(url, stream=True) as response:
    if response.status_code == 200:
        with open("Resources/raster_file.nc", "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:  # filter out keep-alive chunks
                    f.write(chunk)
        print("Download complete.")
    else:
        print(f"Failed to retrieve the file. Status code: {response.status_code}")


Download complete.


In [44]:
# Open NetCDF file
ds = xr.open_dataset("Resources/raster_file.nc")


In [45]:
# Preview variables
ds


<xarray.Dataset>
Dimensions:                    (time: 124, lat: 360, lon: 720, bnds: 2)
Coordinates:
  * time                       (time) datetime64[ns] 1901-07-01 ... 2024-07-01
  * lat                        (lat) float32 -89.75 -89.25 ... 89.25 89.75
  * lon                        (lon) float32 -179.8 -179.2 ... 179.2 179.8
  * bnds                       (bnds) int32 0 1
Data variables:
    timeseries-pr-annual-mean  (time, lat, lon) float32 ...
    lon_bnds                   (lon, bnds) float64 ...
    lat_bnds                   (lat, bnds) float64 ...
Attributes: (12/23)
    wb_truncation_label:   2
    wb_grid_label:         x0.5
    wb_period_label:       1901-2024
    wb_percentile_label:   mean
    wb_type_label:         timeseries
    wb_model_label:        cru
    ...                    ...
    source:                Run ID = 2503051245. Data generated from:pre.25030...
    institution:           Data held at British Atmospheric Data Centre, RAL,...
    title:                 CRU TS4.09 Precipitation
    Conventions:           CF-1.4
    wb_OBS:                 \nwb_creation_date = Sat Apr 26 02:55:23 PM MDT 2...
    wb:                     \nwb_access_citation = Please acknowledge data so...

In [46]:

# Convert to CSV (assumes 2D array or manageable data)
raster_df = ds.to_dataframe().reset_index()

# View df
raster_df.head()


,time,lat,lon,bnds,timeseries-pr-annual-mean,lon_bnds,lat_bnds
0,1901-07-01,-89.75,-179.75,0,NaN,-180.0,-90.0
1,1901-07-01,-89.75,-179.75,1,NaN,-179.5,-89.5
2,1901-07-01,-89.75,-179.25,0,NaN,-179.5,-90.0
3,1901-07-01,-89.75,-179.25,1,NaN,-179.0,-89.5
4,1901-07-01,-89.75,-178.75,0,NaN,-179.0,-90.0


In [48]:
# Look at the same column
raster_df['timeseries-pr-annual-mean'].isna().sum()

47561440

In [49]:
# Still looking
raster_df['timeseries-pr-annual-mean'].notna().sum()

16720160

Most of the entries are NaN, likely the ones outside Canada and the USA.

In [50]:
# Filter for USA
usa_mask = (
    (raster_df['lat'] >= 24.5) & (raster_df['lat'] <= 49.5) &
    (raster_df['lon'] >= -125) & (raster_df['lon'] <= -66.9)
)

# Filter for Canada
canada_mask = (
    (raster_df['lat'] >= 41.7) & (raster_df['lat'] <= 83.1) &
    (raster_df['lon'] >= -141) & (raster_df['lon'] <= -52.6)
)

# Combine both masks
north_america_mask = usa_mask | canada_mask

# Apply filter
raster_df_filtered = raster_df[north_america_mask].copy()

In [51]:
# Look at new raster_df_filtered
raster_df_filtered.head()

,time,lat,lon,bnds,timeseries-pr-annual-mean,lon_bnds,lat_bnds
329980,1901-07-01,24.75,-124.75,0,NaN,-125.0,24.5
329981,1901-07-01,24.75,-124.75,1,NaN,-124.5,25.0
329982,1901-07-01,24.75,-124.25,0,NaN,-124.5,24.5
329983,1901-07-01,24.75,-124.25,1,NaN,-124.0,25.0
329984,1901-07-01,24.75,-123.75,0,NaN,-124.0,24.5


In [52]:
# Look for NaN
raster_df_filtered.isna().sum()

time                               0
lat                                0
lon                                0
bnds                               0
timeseries-pr-annual-mean    1581744
lon_bnds                           0
lat_bnds                           0
dtype: int64

In [53]:
# Look for Not Na
raster_df_filtered.notna().sum()

time                         4621480
lat                          4621480
lon                          4621480
bnds                         4621480
timeseries-pr-annual-mean    3039736
lon_bnds                     4621480
lat_bnds                     4621480
dtype: int64

timeseries column now has not na: 3039736
previously had                  : 16720160

so we've lost a substanial amount of entries, and still have 1581744 Nan 

In [54]:
# Shape of raster_df_filtered
raster_df_filtered.shape

(4621480, 7)

After all that the raster file might not provide what I was hoping. Unfortunate. 

Let's try to look at some Canada crop yield data.

In [56]:
# Load Our World in Data CO2 data ('owid-co2-data.csv')
canada_crops_df = pd.read_csv('Resources/3210035901_databaseLoadingData.csv')
canada_crops_df.head()

,REF_DATE,GEO,DGUID,Harvest disposition,Type of crop,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,1908,Canada,2021A000011124,Production (metric tonnes),Corn for grain,Metric tonnes,214,units,0,v114995731,1.19.11,580600.0,NaN,NaN,NaN,0
1,1909,Canada,2021A000011124,Production (metric tonnes),Corn for grain,Metric tonnes,214,units,0,v114995731,1.19.11,489600.0,NaN,NaN,NaN,0
2,1910,Canada,2021A000011124,Production (metric tonnes),Corn for grain,Metric tonnes,214,units,0,v114995731,1.19.11,363600.0,NaN,NaN,NaN,0
3,1911,Canada,2021A000011124,Production (metric tonnes),Corn for grain,Metric tonnes,214,units,0,v114995731,1.19.11,487400.0,NaN,NaN,NaN,0
4,1912,Canada,2021A000011124,Production (metric tonnes),Corn for grain,Metric tonnes,214,units,0,v114995731,1.19.11,430100.0,NaN,NaN,NaN,0


In [57]:
canada_crops_df.tail()

,REF_DATE,GEO,DGUID,Harvest disposition,Type of crop,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
703,2021,Canada,2021A000011124,Production (metric tonnes),"Wheat, winter remaining",Metric tonnes,214,units,0,v114995756,1.19.3,3227759.0,NaN,NaN,NaN,0
704,2022,Canada,2021A000011124,Production (metric tonnes),"Wheat, winter remaining",Metric tonnes,214,units,0,v114995756,1.19.3,2852803.0,NaN,r,NaN,0
705,2023,Canada,2021A000011124,Production (metric tonnes),"Wheat, winter remaining",Metric tonnes,214,units,0,v114995756,1.19.3,3341757.0,NaN,r,NaN,0
706,2024,Canada,2021A000011124,Production (metric tonnes),"Wheat, winter remaining",Metric tonnes,214,units,0,v114995756,1.19.3,3043363.0,NaN,r,NaN,0
707,2025,Canada,2021A000011124,Production (metric tonnes),"Wheat, winter remaining",Metric tonnes,214,units,0,v114995756,1.19.3,3477773.0,NaN,NaN,NaN,0


Data retrieved from Stats Can : https://www150.statcan.gc.ca/t1/tbl1/en/cv.action?pid=3210035901

Luckily this loaded pretty easily! Now let's try to get one for the USA.

In [58]:
# import API KEY
from Resources import USDA_API_KEY

In [62]:
# Quick USDA API attempt
url = (
    "https://quickstats.nass.usda.gov/api/api_GET/"
    "?key=" + USDA_API_KEY +
    "&source_desc=SURVEY"
    "&sector_desc=CROPS"
    "&commodity_desc=CORN"
    "&statisticcat_desc=YIELD"
    "&agg_level_desc=STATE"
    "&year__GE=2000"
    "&format=CSV"
)

TypeError: can only concatenate str (not "module") to str